In [1]:
import os, re, json, requests, unicodedata
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import pipeline

SAVE_DIR = Path("./hindi_asr")
SAVE_DIR.mkdir(exist_ok=True)
AUDIO_DIR = SAVE_DIR / "audio_16k"
TRANS_DIR = SAVE_DIR / "transcriptions"
AUDIO_DIR.mkdir(exist_ok=True)
TRANS_DIR.mkdir(exist_ok=True)

print("GPU:", torch.cuda.is_available())
print("Save dir:", SAVE_DIR)

GPU: True
Save dir: hindi_asr


In [2]:
df = pd.read_csv("/teamspace/studios/this_studio/FT Data - data.csv")   

def fix_gcp_url(url):
    match = re.search(r'hq_data/hi/(\d+)/(\d+)_(.*)', url)
    if match:
        folder_id, recording_id, suffix = match.groups()
        return f'https://storage.googleapis.com/upload_goai/{folder_id}/{recording_id}_{suffix}'
    return url

df['transcription_url_gcp'] = df['transcription_url_gcp'].apply(fix_gcp_url)
df['rec_url_gcp'] = df['rec_url_gcp'].apply(fix_gcp_url)

print(f"Total recordings: {len(df)}")
print(df['rec_url_gcp'].iloc[0])

Total recordings: 104
https://storage.googleapis.com/upload_goai/967179/825780_audio.wav


In [3]:
TARGET_SR = 16000
valid_ids = []
failed_ids = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Audio"):
    rec_id = str(row['recording_id'])
    dst_path = AUDIO_DIR / f"{rec_id}.wav"

    if dst_path.exists():
        valid_ids.append(rec_id)
        continue

    try:
        r = requests.get(row['rec_url_gcp'], timeout=120, stream=True)
        r.raise_for_status()
        tmp_path = AUDIO_DIR / f"{rec_id}_tmp.wav"
        with open(tmp_path, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

        audio, sr = librosa.load(str(tmp_path), sr=TARGET_SR, mono=True)
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val * 0.95
        sf.write(str(dst_path), audio, TARGET_SR, subtype='PCM_16')
        tmp_path.unlink()
        valid_ids.append(rec_id)
    except Exception as e:
        print(f"  Failed {rec_id}: {e}")
        failed_ids.append(rec_id)

print(f"\nValid: {len(valid_ids)} | Failed: {len(failed_ids)}")

Audio:   0%|          | 0/104 [00:00<?, ?it/s]


Valid: 104 | Failed: 0


In [4]:
def fetch_transcription(url, timeout=30):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, list):
            return " ".join([s["text"].strip() for s in data if s.get("text","").strip()])
        elif isinstance(data, dict):
            return data.get("text") or data.get("transcription", "")
        return None
    except:
        return None

references = {}

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching refs"):
    rec_id = str(row['recording_id'])
    if rec_id not in valid_ids:
        continue
    save_path = TRANS_DIR / f"{rec_id}.txt"
    if save_path.exists():
        references[rec_id] = save_path.read_text(encoding='utf-8')
        continue
    text = fetch_transcription(row['transcription_url_gcp'])
    if text and len(text) > 10:
        save_path.write_text(text, encoding='utf-8')
        references[rec_id] = text

print(f"References fetched: {len(references)}")

Fetching refs:   0%|          | 0/104 [00:00<?, ?it/s]

References fetched: 104


In [ ]:
raw_asr_results = {}
RAW_ASR_DIR = SAVE_DIR / "raw_asr"
RAW_ASR_DIR.mkdir(exist_ok=True)

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-small",
    device = 0 if torch.cuda.is_available() else -1,
    chunk_length_s  = 30,
    generate_kwargs = {"language": "hindi", "task": "transcribe"}
)

for rec_id in tqdm(valid_ids, desc="Running ASR"):
    if rec_id not in references:
        continue

    out_path = RAW_ASR_DIR / f"{rec_id}.txt"
    if out_path.exists():
        raw_asr_results[rec_id] = out_path.read_text(encoding='utf-8')
        continue

    try:
        # Load with librosa 
        audio, sr = librosa.load(
            str(AUDIO_DIR / f"{rec_id}.wav"),
            sr=16000, mono=True
        )
        # Pass as dict with sampling_rate — pipeline handles chunking
        result = asr_pipe({"array": audio, "sampling_rate": sr})
        text = result["text"].strip()
        out_path.write_text(text, encoding='utf-8')
        raw_asr_results[rec_id] = text
    except Exception as e:
        print(f"  ASR failed {rec_id}: {e}")

print(f"ASR done: {len(raw_asr_results)} transcripts")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Running ASR:   0%|          | 0/104 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
pairs = []
for rec_id in raw_asr_results:
    if rec_id in references:
        pairs.append({
            "id": rec_id,
            "raw_asr": raw_asr_results[rec_id],
            "reference": references[rec_id]
        })

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv(SAVE_DIR / "asr_pairs.csv", index=False)
print(f"Paired samples: {len(pairs_df)}")
print("\nSample:")
print("RAW ASR:", pairs_df['raw_asr'].iloc[0][:150])
print("REFERENCE:", pairs_df['reference'].iloc[0][:150])

In [ ]:
UNITS = {
    'शून्य': 0, 'एक': 1, 'दो': 2, 'तीन': 3, 'चार': 4,
    'पाँच': 5, 'पांच': 5, 'छह': 6, 'छः': 6, 'सात': 7,
    'आठ': 8, 'नौ': 9, 'दस': 10, 'ग्यारह': 11, 'बारह': 12,
    'तेरह': 13, 'चौदह': 14, 'पंद्रह': 15, 'सोलह': 16,
    'सत्रह': 17, 'अठारह': 18, 'उन्नीस': 19, 'बीस': 20,
    'इक्कीस': 21, 'बाईस': 22, 'तेईस': 23, 'चौबीस': 24,
    'पच्चीस': 25, 'छब्बीस': 26, 'सत्ताईस': 27, 'अट्ठाईस': 28,
    'उनतीस': 29, 'तीस': 30, 'इकत्तीस': 31, 'बत्तीस': 32,
    'तैंतीस': 33, 'चौंतीस': 34, 'पैंतीस': 35, 'छत्तीस': 36,
    'सैंतीस': 37, 'अड़तीस': 38, 'उनचालीस': 39, 'चालीस': 40,
    'पचास': 50, 'साठ': 60, 'सत्तर': 70, 'अस्सी': 80,
    'नब्बे': 90, 'सौ': 100, 'हज़ार': 1000, 'हजार': 1000,
    'लाख': 100000, 'करोड़': 10000000
}

MULTIPLIERS = {'सौ': 100, 'हज़ार': 1000, 'हजार': 1000,
               'लाख': 100000, 'करोड़': 10000000}

# Idiom/phrase patterns where numbers should NOT be converted
IDIOM_PATTERNS = [
    r'दो-चार',       # दो-चार बातें
    r'चार-पाँच',
    r'दो टूक',       # frank/blunt
    r'तीन-तेरह',     # confusion idiom
    r'नौ-दो ग्यारह', # to flee
    r'एक न एक',
    r'दो और दो',
    r'एक-दो',
    r'चार-छह',
]

def is_idiom(text, pos, word):
    """Check if number word at position is part of an idiom."""
    for pattern in IDIOM_PATTERNS:
        if re.search(pattern, text):
            return True
    return False

def words_to_number(words):
    """Convert a list of Hindi number words to integer."""
    total   = 0
    current = 0
    for word in words:
        if word in MULTIPLIERS:
            mult = MULTIPLIERS[word]
            if mult >= 1000:
                total   = (total + current) * mult if current else total * mult + mult
                current = 0
            else:
                current = current * mult if current else mult
        elif word in UNITS:
            current += UNITS[word]
    return total + current

def normalize_numbers(text):
    """
    Convert Hindi number words to digits.
    Skips idioms and ambiguous phrases.
    """ 
    protected   = {}
    protect_idx = 0
    protected_text = text

    for pattern in IDIOM_PATTERNS:
        matches = list(re.finditer(pattern, protected_text))
        for m in matches:
            placeholder = f"__IDIOM{protect_idx}__"
            protected[placeholder] = m.group()
            protected_text = protected_text[:m.start()] + placeholder + protected_text[m.end():]
            protect_idx += 1
 
    all_nums  = sorted(list(UNITS.keys()) + list(MULTIPLIERS.keys()),
                       key=len, reverse=True)
    num_pattern = '|'.join(re.escape(w) for w in all_nums)
 
    full_pattern = rf'(?:{num_pattern})(?:\s+(?:{num_pattern}))*'

    def replace_number_sequence(match):
        phrase = match.group()
        words  = phrase.split()
        num    = words_to_number(words)
        if num == 0:
            return phrase   
        return str(num)

    result = re.sub(full_pattern, replace_number_sequence, protected_text)
 
    for placeholder, original in protected.items():
        result = result.replace(placeholder, original)

    return result
 
tests = [
    "दो सौ लोग आए थे",
    "एक हज़ार पाँच सौ रुपये",
    "दो-चार बातें करनी थी",
    "पच्चीस साल पहले",
    "तीन सौ चौवन किलोमीटर",
    "नौ-दो ग्यारह हो गए",
]
print("Number Normalization Tests:")
print("-" * 50)
for t in tests:
    print(f"IN: {t}")
    print(f"OUT: {normalize_numbers(t)}")
    print()

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
 
pairs_df['normalized_asr'] = pairs_df['raw_asr'].apply(normalize_numbers)
 
wer_before = wer_metric.compute(
    predictions=pairs_df['raw_asr'].tolist(),
    references=pairs_df['reference'].tolist()
)
wer_after = wer_metric.compute(
    predictions=pairs_df['normalized_asr'].tolist(),
    references=pairs_df['reference'].tolist()
)

print("=" * 50)
print(f"WER before normalization: {wer_before*100:.2f}%")
print(f"WER after normalization:  {wer_after*100:.2f}%")
print(f"Change: {(wer_after - wer_before)*100:+.2f}%")
print("=" * 50)
 
print("\n--- Examples where normalization HELPED ---")
helped = []
for _, row in pairs_df.iterrows():
    w_before = wer_metric.compute(predictions=[row['raw_asr']],    references=[row['reference']])
    w_after  = wer_metric.compute(predictions=[row['normalized_asr']], references=[row['reference']])
    if w_after < w_before:
        helped.append((row['raw_asr'], row['normalized_asr'], row['reference'], w_before, w_after))

for raw, norm, ref, wb, wa in helped[:5]:
    print(f"REF:    {ref[:100]}")
    print(f"BEFORE: {raw[:100]}")
    print(f"AFTER:  {norm[:100]}")
    print(f"WER: {wb*100:.1f}% → {wa*100:.1f}%")
    print()

print("\n--- Examples where normalization HURT ---")
hurt = []
for _, row in pairs_df.iterrows():
    w_before = wer_metric.compute(predictions=[row['raw_asr']],    references=[row['reference']])
    w_after  = wer_metric.compute(predictions=[row['normalized_asr']], references=[row['reference']])
    if w_after > w_before:
        hurt.append((row['raw_asr'], row['normalized_asr'], row['reference'], w_before, w_after))

for raw, norm, ref, wb, wa in hurt[:3]:
    print(f"REF:    {ref[:100]}")
    print(f"BEFORE: {raw[:100]}")
    print(f"AFTER:  {norm[:100]}")
    print(f"WER: {wb*100:.1f}% → {wa*100:.1f}%")
    print()

In [ ]:
# English word detection — handles both Roman script AND Devanagari loanwords

# Common English loanwords written in Devanagari
DEVANAGARI_ENGLISH_LOANWORDS = {
    # Tech
    'कंप्यूटर', 'इंटरनेट', 'मोबाइल', 'फोन', 'वीडियो', 'ऑनलाइन',
    'सॉफ्टवेयर', 'हार्डवेयर', 'ऐप', 'वेबसाइट', 'ईमेल', 'चैट',
    'स्क्रीन', 'कीबोर्ड', 'माउस', 'लैपटॉप', 'टैबलेट', 'स्मार्टफोन',
    # Work/career
    'इंटरव्यू', 'जॉब', 'ऑफिस', 'मैनेजर', 'टीम', 'प्रोजेक्ट',
    'मीटिंग', 'प्रेजेंटेशन', 'रिपोर्ट', 'टारगेट', 'बजट',
    # Daily life
    'बस', 'ट्रेन', 'होटल', 'रेस्टोरेंट', 'शॉपिंग', 'मॉल',
    'पार्क', 'स्कूल', 'कॉलेज', 'यूनिवर्सिटी', 'क्लास',
    # Finance
    'बैंक', 'लोन', 'क्रेडिट', 'डेबिट', 'इन्वेस्टमेंट', 'स्टॉक',
    # Common adjectives/verbs used in Hinglish
    'प्रॉब्लम', 'सॉल्व', 'मैनेज', 'हैंडल', 'अपडेट', 'डाउनलोड',
    'सीरियस', 'नॉर्मल', 'रेगुलर', 'स्पेशल', 'फाइनल', 'टोटल',
    # Media/entertainment
    'मूवी', 'सीरीज', 'चैनल', 'न्यूज़', 'सोशल मीडिया', 'पोस्ट',
    'लाइव', 'स्ट्रीम', 'कंटेंट', 'क्रिएटर',
}

def detect_english_words(text):
    """
    Tag English words in Hindi text.
    Handles:
    1. Roman script English words
    2. Devanagari transliterated English loanwords
    Returns tagged text and list of detected English words.
    """
    words = text.split()
    tagged = []
    en_words = []

    i = 0
    while i < len(words):
        word = words[i]
        clean_word = re.sub(r'[^\u0900-\u097Fa-zA-Z]', '', word)

        # Check 1: Roman script word (Latin characters)
        if re.match(r'^[a-zA-Z]+$', clean_word):
            tagged.append(f"[EN]{word}[/EN]")
            en_words.append(word)

        # Check 2: Mixed script (some Hindi some Roman)
        elif re.search(r'[a-zA-Z]', word) and re.search(r'[\u0900-\u097F]', word):
            tagged.append(f"[EN]{word}[/EN]")
            en_words.append(word)

        # Check 3: Known Devanagari loanword (single word)
        elif clean_word in DEVANAGARI_ENGLISH_LOANWORDS:
            tagged.append(f"[EN]{word}[/EN]")
            en_words.append(word)

        # Check 4: Two-word Devanagari loanword (e.g. सोशल मीडिया)
        elif i + 1 < len(words):
            two_word = clean_word + ' ' + re.sub(r'[^\u0900-\u097F]', '', words[i+1])
            if two_word in DEVANAGARI_ENGLISH_LOANWORDS:
                tagged.append(f"[EN]{word} {words[i+1]}[/EN]")
                en_words.append(f"{word} {words[i+1]}")
                i += 2
                continue
            else:
                tagged.append(word)
        else:
            tagged.append(word)

        i += 1

    return " ".join(tagged), en_words


# Test
test_sentences = [
    "मेरा interview बहुत अच्छा गया और मुझे job मिल गई",
    "आज मीटिंग में प्रेजेंटेशन देनी है",
    "मेरा मोबाइल चार्ज नहीं हो रहा",
    "उसने online shopping की और product return किया",
    "कंप्यूटर में software update आया है",
]

print("English Word Detection Tests:")
print("-" * 60)
for s in test_sentences:
    tagged, en_words = detect_english_words(s)
    print(f"IN: {s}")
    print(f"OUT: {tagged}")
    print(f"EN words: {en_words}")
    print()

In [ ]:
pairs_df['tagged_asr'] = pairs_df['raw_asr'].apply(lambda x: detect_english_words(x)[0])
pairs_df['english_words'] = pairs_df['raw_asr'].apply(lambda x: detect_english_words(x)[1])
pairs_df['en_word_count'] = pairs_df['english_words'].apply(len)

print(f"Transcripts with English words: {(pairs_df['en_word_count'] > 0).sum()}")
print(f"Avg English words per transcript: {pairs_df['en_word_count'].mean():.2f}")
print(f"Max English words in one transcript: {pairs_df['en_word_count'].max()}")

print("\n--- Sample tagged outputs ---")
sample = pairs_df[pairs_df['en_word_count'] > 0].head(5)
for _, row in sample.iterrows():
    print(f"RAW: {row['raw_asr'][:150]}")
    print(f"TAGGED: {row['tagged_asr'][:150]}")
    print(f"EN: {row['english_words']}")
    print()

In [ ]:
def full_pipeline(text):
    """Apply both operations in sequence."""
    normalized = normalize_numbers(text)
    tagged, en_words = detect_english_words(normalized)
    return {
        "original": text,
        "normalized": normalized,
        "tagged": tagged,
        "en_words": en_words
    }
 
results = pairs_df['raw_asr'].apply(full_pipeline)
pairs_df['final_output'] = results.apply(lambda x: x['tagged'])
 
pairs_df.to_csv(SAVE_DIR / "q2_pipeline_output.csv", index=False)
print("Pipeline complete. Output saved.")
print(f"Total processed: {len(pairs_df)}")
 
print("\n=== Pipeline Examples ===")
for _, row in pairs_df.head(3).iterrows():
    r = full_pipeline(row['raw_asr'])
    print(f"ORIGINAL: {r['original'][:120]}")
    print(f"NORMALIZED: {r['normalized'][:120]}")
    print(f"TAGGED: {r['tagged'][:120]}")
    print(f"EN WORDS: {r['en_words']}")
    print()

In [ ]:
print("=" * 60)
print("PIPELINE SUMMARY REPORT")
print("=" * 60)

print(f"\nDataset: {len(pairs_df)} audio recordings")
print(f"Total duration: {df['duration'].sum()/3600:.1f} hours")

print(f"\n--- a) Number Normalization ---")
print(f"WER before: {wer_before*100:.2f}%")
print(f"WER after: {wer_after*100:.2f}%")
print(f"Change: {(wer_after - wer_before)*100:+.2f}%")
print(f"Helped on: {len(helped)} transcripts")
print(f"Hurt on: {len(hurt)} transcripts")

print(f"\n--- b) English Word Detection ---")
print(f"Transcripts with English words: {(pairs_df['en_word_count'] > 0).sum()}")
print(f"Total English words detected: {pairs_df['en_word_count'].sum()}")
print(f"Avg per transcript: {pairs_df['en_word_count'].mean():.2f}")

# Most common English words
from collections import Counter
all_en = [w for sublist in pairs_df['english_words'] for w in sublist]
top_en = Counter(all_en).most_common(10)
print(f"\nTop 10 English words detected:")
for word, count in top_en:
    print(f"  {word}: {count}")

print("\nDone! Output saved to q2_pipeline_output.csv")